In [1]:
import pandas as pd

url = "https://football-data.co.uk/mmz4281/2223/E0.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

(380, 106)


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,AvgC<2.5,AHCh,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA
0,E0,05/08/2022,20:00,Crystal Palace,Arsenal,0,2,A,0,1,...,1.76,0.50,2.09,1.84,2.04,1.88,2.09,1.88,2.03,1.85
1,E0,06/08/2022,12:30,Fulham,Liverpool,2,2,D,1,0,...,2.73,1.75,1.90,2.03,1.91,2.02,2.01,2.06,1.89,1.99
2,E0,06/08/2022,15:00,Bournemouth,Aston Villa,2,0,H,1,0,...,1.76,0.50,1.93,2.00,1.93,2.00,1.94,2.04,1.88,2.00
3,E0,06/08/2022,15:00,Leeds,Wolves,2,1,H,1,1,...,1.87,-0.25,2.08,1.85,2.10,1.84,2.14,1.87,2.08,1.81
4,E0,06/08/2022,15:00,Newcastle,Nott'm Forest,2,0,H,0,0,...,1.89,-1.00,1.97,1.96,1.99,1.93,2.19,1.97,2.03,1.86


In [2]:
url = "https://football-data.co.uk/mmz4281/2223/E0.csv"
df = pd.read_csv(url, encoding="latin-1")

keep = ["FTR",
        "B365H", "B365D", "B365A",
        "BWH", "BWD", "BWA",
        "PSH", "PSD", "PSA",
        "WHH", "WHD", "WHA",
        "VCH", "VCD", "VCA"]

# check that every requested column exists in the file
missing = [c for c in keep if c not in df.columns]
if missing:
    print("Columns not found:", missing)

df = df[[c for c in keep if c in df.columns]].copy()

print(df.shape)
df.head()

# count missing values per column before cleaning
print(df.isna().sum())
print("Rows before:", len(df))

# drop any row with at least one missing value
df = df.dropna().reset_index(drop=True)

print("Rows after:", len(df))

(380, 16)
FTR      0
B365H    0
B365D    0
B365A    0
BWH      0
BWD      0
BWA      0
PSH      0
PSD      0
PSA      0
WHH      0
WHD      0
WHA      0
VCH      0
VCD      0
VCA      0
dtype: int64
Rows before: 380
Rows after: 380


In [3]:
bookmakers = {"Bet365": "B365", "Bwin": "BW", "Pinnacle": "PS",
              "William Hill": "WH", "VC Bet": "VC"}

# 1. one-hot outcome matrix, columns ordered H, D, A
y = pd.get_dummies(df["FTR"])[["H", "D", "A"]].to_numpy(dtype=float)

# 2. convert decimal odds to probabilities (proportional normalisation)
def odds_to_probs(odds):
    raw = 1 / odds                                   # implied probabilities (sum > 1)
    return raw / raw.sum(axis=1, keepdims=True)      # remove the overround

# 3. multiclass Brier score, one value per match
def brier(probs, outcomes):
    return ((probs - outcomes) ** 2).sum(axis=1)

# 4. score each bookmaker
scores = {}
for name, p in bookmakers.items():
    odds = df[[f"{p}H", f"{p}D", f"{p}A"]].to_numpy(dtype=float)
    scores[name] = brier(odds_to_probs(odds), y)

# 5. summary table, best (lowest) first
summary = (pd.DataFrame({"Brier score": {k: v.mean() for k, v in scores.items()}})
           .sort_values("Brier score"))
print(f"Matches scored: {len(df)}")
print(summary.round(4))

Matches scored: 380
              Brier score
Bwin               0.5731
William Hill       0.5735
Bet365             0.5735
VC Bet             0.5740
Pinnacle           0.5740
